# Student Performance Prediction Project

## 1. Data Initialization

In [84]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import xgboost as xgb

file_path = 'archive (1).zip'
df = pd.read_csv(file_path)
print("Sample DataFrame Head:")
display(df.head())
print(f"Total number of rows: {len(df)}")

Sample DataFrame Head:


,student_id,age,gender,city_type,study_hours_per_day,deep_work_sessions,assignment_completion_rate,attendance_percentage,social_media_hours,doomscrolling_before_sleep,...,family_support,financial_stress,learning_style,career_goal,productivity_after_midnight,revision_efficiency,burnout_risk,consistency_score,final_exam_score,performance_category
0,0,21,Female,Semi-Urban,3.2,7,100,70,3.8,0,...,10,6,Audio,Engineering,9,1,10,3,32,Low
1,1,19,Female,Semi-Urban,3.9,2,46,70,2.6,1,...,2,6,Practical,Business,6,10,4,6,59,Medium
2,2,16,Female,Urban,4.3,7,54,57,4.3,1,...,1,9,Visual,Engineering,7,9,8,1,34,Low
3,3,19,Male,Semi-Urban,5.3,1,78,90,1.7,0,...,5,10,Reading,Medical,7,7,9,10,60,Medium
4,4,17,Female,Urban,4.1,3,100,81,2.6,0,...,5,6,Visual,Medical,4,10,7,9,77,High


Total number of rows: 3000


## 2. Data Cleaning

In [ ]:

df.drop_duplicates(inplace=True)
df.fillna(df.mean(numeric_only=True), inplace=True)

print("Data Cleaning Completed")
print("DataFrame after cleaning (head):")
display(df.head())

## 3. Feature Engineering

In [ ]:
# Calculate a new PerformanceIndex using the actual columns from your dataset
# Since attendance_percentage and assignment_completion_rate are both on a 0-100 scale, we can average them.
df["PerformanceIndex"] = (
    df["attendance_percentage"] +
    df["assignment_completion_rate"]
) / 2


print("DataFrame with engineered 'PerformanceIndex', 'social_media_hours','final_exam_score','study_hours_per_day','family_support' features:")
display(df[["attendance_percentage", "assignment_completion_rate", "PerformanceIndex", "social_media_hours", "family_support","study_hours_per_day"]].head())

## 4. Exploratory Data Analysis (EDA)

### Basic Information and Statistics

In [ ]:

print("DataFrame Info:")
df.info()
print("\nDescriptive Statistics:")
display(df.describe())

### Visualizing Data Distributions and Relationships

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df['final_exam_score'], kde=True, bins=5)
plt.title('Distribution of Final Exam Scores')
plt.xlabel('Final Exam Score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(x='study_hours_per_day', y='final_exam_score', data=df)
plt.title('Study Hours vs. Final Exam Score')
plt.xlabel('Study Hours Per Day')
plt.ylabel('Final Exam Score')
plt.grid(linestyle='--', alpha=0.7)
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PerformanceIndex', y='final_exam_score', data=df)
plt.title('Performance Index vs. Final Exam Score')
plt.xlabel('Performance Index')
plt.ylabel('Final Exam Score')
plt.grid(linestyle='--', alpha=0.7)
plt.show()

plt.figure(figsize=(8, 6))
sns.boxplot(y=df['final_exam_score'])
plt.title('Boxplot of Final Exam Scores')
plt.ylabel('Final Exam Score')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Correlation Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select only the primary key numerical features of interest to keep the plot neat
key_features = [
    "attendance_percentage",
    "assignment_completion_rate",
    "study_hours_per_day",
    "social_media_hours",
    "family_support",
    "final_exam_score"
]

plt.figure(figsize=(8, 6))
sns.heatmap(
    df[key_features].corr(),
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    square=True
)
plt.title('Correlation Matrix of Key Features', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Regression Task: Predicting FinalMarks

### Data Splitting for Regression

In [ ]:
# Drop PerformanceIndex to eliminate absolute multicollinearity and data leakage
X = df[
    [
        "attendance_percentage",
        "assignment_completion_rate",
        "study_hours_per_day",
        "social_media_hours",
        "family_support"
    ]
]
y = df["final_exam_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data Shape (Regression):", X_train.shape)
print("Testing Data Shape (Regression):", X_test.shape)

### 5.1. Linear Regression Model

In [ ]:
def train_evaluate_regression_model(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{model_name} Model Training Completed")
    print(f"{model_name} - Mean Absolute Error: {mae:.2f}")
    print(f"{model_name} - R2 Score: {r2:.2f}")
    return mae, r2

linear_model = LinearRegression()
mae_linear, r2_linear = train_evaluate_regression_model(
    linear_model,
    X_train,
    y_train,
    X_test,
    y_test,
    "Linear Regression"
)

### 5.2. Random Forest Regressor Model

In [ ]:
rf_model = RandomForestRegressor(random_state=42)
rf_mae, rf_r2 = train_evaluate_regression_model(rf_model, X_train, y_train, X_test, y_test, "Random Forest")

### 5.3. XGBoost Regressor Model

In [ ]:
xgb_model = xgb.XGBRegressor(random_state=42)
xgb_mae, xgb_r2 = train_evaluate_regression_model(xgb_model, X_train, y_train, X_test, y_test, "XGBoost")

### 5.4. Prediction Functionality for New Students (using Linear Regression as the best model for this example)

In [85]:
# Prompting inputs matching the actual feature schema of our trained linear model
attendance_pct = float(input("Enter Attendance Percentage (0-100): "))
assignment_rate = float(input("Enter Assignment Completion Rate (0-100): "))

# Ensure study hours takes only 0-10 hours
while True:
    try:
        study_hours = float(input("Enter Study Hours Per Day (0-10): "))
        if 0 <= study_hours <= 10:
            break
        else:
            print("Invalid input. Study hours must be between 0 and 10.")
    except ValueError:
        print("Please enter a valid numeric value.")

# Ensure social media hours takes only 0-10 hours
while True:
    try:
        social_media = float(input("Enter Social Media Hours (0-10): "))
        if 0 <= social_media <= 10:
            break
        else:
            print("Invalid input. Social media hours must be between 0 and 10.")
    except ValueError:
        print("Please enter a valid numeric value.")

# Ensure family support is rated on a 0-10 scale
while True:
    try:
        family_support = float(input("Enter Family Support rating (0-10): "))
        if 0 <= family_support <= 10:
            break
        else:
            print("Invalid input. Family support must be between 0 and 10.")
    except ValueError:
        print("Please enter a valid numeric value.")

# Creating the new student DataFrame WITHOUT the redundant PerformanceIndex
new_student = pd.DataFrame(
    [[
        attendance_pct,
        assignment_rate,
        study_hours,
        social_media,
        family_support
    ]],
    columns=[
        "attendance_percentage",
        "assignment_completion_rate",
        "study_hours_per_day",
        "social_media_hours",
        "family_support"
    ]
)

# Run the prediction using our trained model
predicted_marks_array = linear_model.predict(new_student)
predicted_mark_scalar = predicted_marks_array[0]
predicted_mark_clipped = max(0, min(100, predicted_mark_scalar))

print(f"\nRaw predicted value: {predicted_mark_scalar:.2f}")
print("Predicted Final Marks =", round(predicted_mark_clipped, 2))

if predicted_mark_clipped >= 40:
    print("Status: Pass")
else:
    print("Status: Fail")

Enter Attendance Percentage (0-100): 87
Enter Assignment Completion Rate (0-100): 76
Enter Study Hours Per Day (0-10): 8
Enter Social Media Hours (0-10): 0
Enter Family Support rating (0-10): 9

Raw predicted value: 81.04
Predicted Final Marks = 81.04
Status: Pass


## 6. Classification Task: Predicting Pass/Fail Status

### Create New Target Variable (`PassFail`)

In [ ]:
df['PassFail'] = df['final_exam_score'].apply(lambda x: 'Pass' if x > 40 else 'Fail')

print("DataFrame with new 'PassFail' column:")
display(df[['final_exam_score', 'PassFail']].head())
print('\nValue Counts for PassFail:')
print(df['PassFail'].value_counts())

### Data Splitting for Classification

In [ ]:
X_clf = df[[
    "attendance_percentage",
    "assignment_completion_rate",
    "study_hours_per_day",
    "social_media_hours",
    "family_support"
]]
y_clf = df['PassFail']

label_encoder = LabelEncoder()
y_clf_encoded = label_encoder.fit_transform(y_clf)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf_encoded,
    test_size=0.2,
    random_state=1,
    stratify=y_clf_encoded
)

print("Original Pass/Fail distribution:", y_clf.value_counts(normalize=True))
print("\nTraining Data Shape for Classification:", X_train_clf.shape)
print("Testing Data Shape for Classification:", X_test_clf.shape)
print("Encoded target classes:", label_encoder.classes_)

### 6.1. Logistic Regression Classifier

In [ ]:
def train_evaluate_classification_model(model, X_train, y_train, X_test, y_test, model_name, label_encoder):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"{model_name} Metrics:")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1-Score: {f1:.2f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    return accuracy, precision, recall, f1

log_reg_model = LogisticRegression(random_state=42, solver='liblinear')
log_reg_accuracy, log_reg_precision, log_reg_recall, log_reg_f1 = \
    train_evaluate_classification_model(log_reg_model, X_train_clf, y_train_clf, X_test_clf, y_test_clf, "Logistic Regression", label_encoder)

### 6.2. Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_accuracy, dt_precision, dt_recall, dt_f1 = \
    train_evaluate_classification_model(dt_model, X_train_clf, y_train_clf, X_test_clf, y_test_clf, "Decision Tree Classifier", label_encoder)

## 7. Final Model Comparison

### Regression Model Comparison

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

def perform_regression_cross_validation(model, X, y, model_name):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mae_scores = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    r2_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
    # Optionally print scores for debugging or detailed view
    # print(f"{model_name} Cross-Validation MAE Scores:", [round(score, 2) for score in mae_scores])
    # print(f"{model_name} Average MAE:", round(mae_scores.mean(), 2))
    # print(f"{model_name} Cross-Validation R2 Scores:", [round(score, 2) for score in r2_scores])
    # print(f"{model_name} Average R2 Score:", round(r2_scores.mean(), 2))
    return mae_scores.mean(), r2_scores.mean()


# Calculate cross-validation scores for Linear Regression
average_mae_lr_cv, average_r2_lr_cv = perform_regression_cross_validation(linear_model, X, y, "Linear Regression")

regression_models = {
    'Linear Regression': {
        'MAE (Test)': mae_linear, 'R2 Score (Test)': r2_linear,
        'MAE (CV)': average_mae_lr_cv, 'R2 Score (CV)': average_r2_lr_cv
    },
    'Random Forest Regressor': {'MAE (Test)': rf_mae, 'R2 Score (Test)': rf_r2},
    'XGBoost Regressor': {'MAE (Test)': xgb_mae, 'R2 Score (Test)': xgb_r2}
}

regression_df = pd.DataFrame.from_dict(regression_models, orient='index')

print("Regression Model Performance Comparison:")
display(regression_df.round(2))

In [ ]:
# Accessing metrics directly from the current kernel session variables
classification_models = {
    'Logistic Regression': {
        'Accuracy (Test)': globals().get('log_reg_accuracy', None),
        'Precision (Test)': globals().get('log_reg_precision', None),
        'Recall (Test)': globals().get('log_reg_recall', None),
        'F1-Score (Test)': globals().get('log_reg_f1', None),
        'Accuracy (CV)': globals().get('average_accuracy_lr_cv', None),
        'Precision (CV)': globals().get('average_precision_lr_cv', None),
        'Recall (CV)': globals().get('average_recall_lr_cv', None),
        'F1-Score (CV)': globals().get('average_f1_lr_cv', None)
    },
    'Decision Tree Classifier': {
        'Accuracy (Test)': globals().get('dt_accuracy', None),
        'Precision (Test)': globals().get('dt_precision', None),
        'Recall (Test)': globals().get('dt_recall', None),
        'F1-Score (Test)': globals().get('dt_f1', None),
        'Accuracy (CV)': globals().get('average_accuracy_dt_cv', None),
        'Precision (CV)': globals().get('average_precision_dt_cv', None),
        'Recall (CV)': globals().get('average_recall_dt_cv', None),
        'F1-Score (CV)': globals().get('average_f1_dt_cv', None)
    }
}

classification_df = pd.DataFrame.from_dict(classification_models, orient='index')

print("Classification Model Performance Comparison:")
display(classification_df.round(2))

## 8. Cross-Validation

### 8.1. Cross-Validation for Regression

In [ ]:

def perform_regression_cross_validation(model, X, y, model_name):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mae_scores = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    r2_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')

    print(f"{model_name} Cross-Validation MAE Scores:", [round(score, 2) for score in mae_scores])
    print(f"{model_name} Average MAE:", round(mae_scores.mean(), 2))
    print(f"{model_name} Cross-Validation R2 Scores:", [round(score, 2) for score in r2_scores])
    print(f"{model_name} Average R2 Score:", round(r2_scores.mean(), 2))
    return mae_scores.mean(), r2_scores.mean()
print("\n--- Linear Regression Cross-Validation ---")
average_mae_lr_cv, average_r2_lr_cv = perform_regression_cross_validation(linear_model, X, y, "Linear Regression")

### 8.2. Cross-Validation for Classification (Logistic Regression and Decision Tree)

In [ ]:
def perform_classification_cross_validation(model, X, y_encoded, model_name):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    accuracy_scores = cross_val_score(model, X, y_encoded, cv=skf, scoring='accuracy')
    precision_scores = cross_val_score(model, X, y_encoded, cv=skf, scoring='precision')
    recall_scores = cross_val_score(model, X, y_encoded, cv=skf, scoring='recall')
    f1_scores = cross_val_score(model, X, y_encoded, cv=skf, scoring='f1')

    print(f"\n{model_name} Cross-Validation Metrics:")
    print("Accuracy Scores:", [round(score, 2) for score in accuracy_scores])
    print("Average Accuracy:", round(accuracy_scores.mean(), 2))
    print("Precision Scores:", [round(score, 2) for score in precision_scores])
    print("Average Precision:", round(precision_scores.mean(), 2))
    print("Recall Scores:", [round(score, 2) for score in recall_scores])
    print("Average Recall:", round(recall_scores.mean(), 2))
    print("F1-Score Scores:", [round(score, 2) for score in f1_scores])
    print("Average F1-Score:", round(f1_scores.mean(), 2))
    return accuracy_scores.mean(), precision_scores.mean(), recall_scores.mean(), f1_scores.mean()

print("\n--- Logistic Regression Cross-Validation ---")
average_accuracy_lr_cv, average_precision_lr_cv, average_recall_lr_cv, average_f1_lr_cv = \
    perform_classification_cross_validation(log_reg_model, X_clf, y_clf_encoded, "Logistic Regression")

print("\n--- Decision Tree Classifier Cross-Validation ---")
average_accuracy_dt_cv, average_precision_dt_cv, average_recall_dt_cv, average_f1_dt_cv = \
    perform_classification_cross_validation(dt_model, X_clf, y_clf_encoded, "Decision Tree Classifier")